# 01-Data Generation



In [1]:
from faker import Faker
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

fake = Faker('es_ES')


In [2]:
hoy = datetime.now()
hace_un_anio = hoy - timedelta(days=365)
hace_un_anio

datetime.datetime(2025, 5, 13, 22, 38, 35, 445386)

## Generate customers
- customer_id
- company_name
- industry
- country
- plan_type
- mrr
- contract_start
- contract_end
- churned
- churn_date
- employee_count
- health_score

In [3]:
def generar_cliente():
    rangos_mmr = {
        "Starter": (99, 299),
        "Pro": (300, 999),
        "Enterprise": (1000, 5000)
    }
    plan_type = random.choice(["Starter", "Pro", "Enterprise"])
    churned = fake.boolean()
    contract_start = fake.date_between(start_date="-4y", end_date='today')
    contract_end = contract_start + timedelta(days=30*random.choice([1,3,6,12]))
    if churned == True: 
        churned_date = fake.date_between(start_date=contract_start, end_date=contract_end)
        health_score = random.randint(0,50)
    else:
        churned_date = None
        health_score = random.randint(40,100)


    return {
        "customer_id":      fake.uuid4(),
        "company_name":     fake.company(),
        "industry":         random.choice(["Tech", "Retail", "Finance", "Health"]),
        "country":          fake.country(),
        "plan_type":        plan_type,
        "mrr":              random.randint(*rangos_mmr[plan_type]),
        "contract_start":   contract_start,
        "contract_end":     contract_end,
        "churned":          churned,
        "churned_date":     churned_date,
        "employee_count":   random.randint(10,5000),
        "health_score":     health_score
    }

print(generar_cliente())

{'customer_id': '56459501-7974-465d-83a2-e9a93c2c9cce', 'company_name': 'Lumbreras y Miranda S.C.P', 'industry': 'Finance', 'country': 'Serbia', 'plan_type': 'Pro', 'mrr': 739, 'contract_start': datetime.date(2023, 7, 23), 'contract_end': datetime.date(2024, 7, 17), 'churned': False, 'churned_date': None, 'employee_count': 2892, 'health_score': 93}


In [4]:
customers = [generar_cliente() for _ in range(3000)]

In [5]:
df_customers = pd.DataFrame(data=customers)
df_customers

,customer_id,company_name,industry,country,plan_type,mrr,contract_start,contract_end,churned,churned_date,employee_count,health_score
0,9dc8e7a4-7c12-42c8-9397-ebdb9348630b,Inmobiliaria Ayala S.A.,Health,Luxemburgo,Starter,240,2025-08-15,2025-11-13,True,2025-11-06,835,22
1,260f0370-048a-4627-8c2e-ae519f707ff2,Marcela Olivé Romero S.A.,Retail,San Marino,Enterprise,3425,2023-11-25,2023-12-25,True,2023-12-22,1368,25
2,3edd9bf2-5a03-4757-90f5-927a2af35aa7,Desarrollo Española S.C.P,Retail,Liechtenstein,Pro,904,2024-09-28,2025-03-27,True,2024-09-30,844,41
3,59c2779c-9fc4-4e2c-9bb7-39adbe932910,Industrias Iberia S.Coop.,Tech,Bahrein,Enterprise,4467,2025-05-02,2025-07-31,False,None,1015,50
4,e5c33a52-e202-48f1-881b-1d35c84c6f91,Fábrica Rozas y asociados S.A.,Health,Tayikistán,Enterprise,1716,2024-09-02,2024-12-01,False,None,2900,90
...,...,...,...,...,...,...,...,...,...,...,...,...
2995,4a3d4934-1712-48ac-be11-a11f72ffc49a,Transportes Valcárcel S.Coop.,Retail,Haití,Enterprise,3639,2022-10-18,2023-01-16,False,None,4492,77
2996,efba63c5-5bda-4a50-98a7-f8c90263509b,Tatiana Berenguer Arregui S.A.,Health,Granada,Starter,154,2024-05-18,2025-05-13,True,2024-12-01,3626,5
2997,89c336dd-22e4-4cdd-9859-40da555e9860,Talleres Cornejo S.A.,Tech,Mauricio,Enterprise,4663,2022-06-22,2022-07-22,False,None,1961,94
2998,13543aa3-991e-439d-ba13-e90aea21156b,Servicios Moll S.A.,Tech,Turquía,Pro,564,2025-10-10,2026-10-05,False,None,4203,43


In [6]:
l_cust = int(len(df_customers))
indices = np.random.choice(l_cust, size=int(l_cust*0.05), replace=False)
df_customers.loc[indices, "country"] = None

In [7]:
indices = np.random.choice(l_cust, size=int(l_cust*0.08), replace=False)
df_customers.loc[indices, "employee_count"] = np.nan

In [8]:
indices = np.random.choice(l_cust, size=int(l_cust*0.03), replace=False)
df_customers.loc[indices, "industry"] = None

In [9]:
df_customers.isna().sum()

customer_id          0
company_name         0
industry            90
country            150
plan_type            0
mrr                  0
contract_start       0
contract_end         0
churned              0
churned_date      1467
employee_count     240
health_score         0
dtype: int64

In [10]:
df_customers.to_csv('../data/raw/customers.csv', index=False)

## Generate Subscriptions
- sub_id
- customer_id
- plan
- billing_cycle
- start_date
- end_date
- discount_pct
- payment_method
- last_payment_status

In [11]:
def generar_suscripcion(cliente: dict):
    if 12 > ((cliente["contract_end"] - cliente["contract_start"]).days / 30):
        billing_cycle = "Mensual"
    else: 
        billing_cycle = "Anual"

    r = random.randint(0,100)
    if cliente["churned"] == True:
        
        if r < 40:
            last_payment_status = "failed"
        elif 40 <= r and r <= 80:
            last_payment_status = "pending"
        else:
            last_payment_status = "success"
    else:
        if r < 70:
            last_payment_status = "success"
        elif 70 <= r and r <= 85:
            last_payment_status = "pending"
        else:
            last_payment_status = "failed"
    
    r = random.randint(0,100)
    if r < 5:
        discount_pct = 30
    elif 5 <= r and r <= 15:
        discount_pct = random.choice([20,30])
    else:
        discount_pct = 0

    return {
        "sub_id":               fake.uuid4(),
        "customer_id":          cliente["customer_id"],
        "plan":                 cliente["plan_type"],
        "billing_cycle":        billing_cycle,
        "start_date":           cliente["contract_start"],
        "end_date":             cliente["contract_end"],
        "discount_pct":         discount_pct,       
        "payment_method":       random.choice(["tarjeta de credito","transferencia bancaria","PayPal"]),      
        "last_payment_status":  last_payment_status
    }

In [12]:
subscriptions = [generar_suscripcion(cust) for cust in df_customers.to_dict('records')]

In [13]:
df_subscriptions = pd.DataFrame(subscriptions)
df_subscriptions

,sub_id,customer_id,plan,billing_cycle,start_date,end_date,discount_pct,payment_method,last_payment_status
0,b63e0779-b308-44f7-a96f-28101e94efd8,9dc8e7a4-7c12-42c8-9397-ebdb9348630b,Starter,Mensual,2025-08-15,2025-11-13,0,PayPal,failed
1,b8d3733a-7ddd-4131-8fdb-ce444350db8b,260f0370-048a-4627-8c2e-ae519f707ff2,Enterprise,Mensual,2023-11-25,2023-12-25,0,PayPal,success
2,c677041b-4d00-4615-8ee8-5fe6d18a455e,3edd9bf2-5a03-4757-90f5-927a2af35aa7,Pro,Mensual,2024-09-28,2025-03-27,0,tarjeta de credito,pending
3,68e5c44e-18aa-4acf-84d5-67bcc266ccdd,59c2779c-9fc4-4e2c-9bb7-39adbe932910,Enterprise,Mensual,2025-05-02,2025-07-31,0,PayPal,pending
4,294e08c6-e6dd-4959-9c65-bcae40ee86eb,e5c33a52-e202-48f1-881b-1d35c84c6f91,Enterprise,Mensual,2024-09-02,2024-12-01,0,transferencia bancaria,failed
...,...,...,...,...,...,...,...,...,...
2995,d18828fb-a3aa-401e-95bd-39562fbf600c,4a3d4934-1712-48ac-be11-a11f72ffc49a,Enterprise,Mensual,2022-10-18,2023-01-16,30,tarjeta de credito,success
2996,cbbd1f88-b22f-4254-a901-689d26b39e69,efba63c5-5bda-4a50-98a7-f8c90263509b,Starter,Anual,2024-05-18,2025-05-13,20,PayPal,pending
2997,05aa6bfc-771e-49ce-aa81-b944fa314608,89c336dd-22e4-4cdd-9859-40da555e9860,Enterprise,Mensual,2022-06-22,2022-07-22,0,tarjeta de credito,success
2998,cfce22be-0618-415b-8e96-fb9387264b62,13543aa3-991e-439d-ba13-e90aea21156b,Pro,Anual,2025-10-10,2026-10-05,0,PayPal,failed


In [14]:
df_subscriptions.to_csv('../data/raw/subscriptions.csv', index=False)

## Usage Logs
- log_id
- customer_id
- user_id
- event_date
- feature_used
- session_duration_min
- login_count
- actions_count
- device_type

In [15]:
def generar_logs_cliente(cliente: dict):
    logs = []
    if cliente["churned"] == True: 
        n_logs = random.randint(10,80)
    else:
        n_logs = random.randint(50,100)

    if cliente["plan_type"] == "Starter":
        users = [fake.uuid4() for _ in range(np.random.randint(1,6))]
    elif cliente["plan_type"] == "Pro":
        users = [fake.uuid4() for _ in range(np.random.randint(5,21))]
    elif cliente["plan_type"] == "Enterprise":
        users = [fake.uuid4() for _ in range(np.random.randint(20,51))]

    modules = ["dashboard","task_manager","reports","integrations","billing","team_settings","calendar","notifications"]

    

    for _ in range(n_logs):
        if cliente["churned"] == True: 
            event_date = fake.date_between(start_date=cliente["contract_start"],end_date=cliente["churned_date"])
            limit_day = (cliente['churned_date'] - event_date).days
            if  limit_day <= 28 and limit_day > 21:
                session_duration_min = random.randint(1,31)
                actions_count = random.randint(1,21)
            elif  limit_day <= 20 and limit_day > 14:
                session_duration_min = random.randint(1,23)
                actions_count = random.randint(1,16)
            elif  limit_day <= 14 and limit_day > 7:
                session_duration_min = random.randint(1,16)
                actions_count = random.randint(1,11)
            elif  limit_day <= 7 and limit_day > 0:
                session_duration_min = random.randint(1,9)
                actions_count = random.randint(1,5)
            else:
                session_duration_min = random.randint(1,41)
                actions_count = random.randint(1,31)
        else:
            event_date = fake.date_between(start_date=cliente["contract_start"],end_date=cliente["contract_end"])
            session_duration_min = random.randint(10,120)
            actions_count = random.randint(10,101)
            

        logs.append({
            "log_id":               fake.uuid4(),
            "customer_id":          cliente["customer_id"],
            "user_id":              random.choice(users),
            "event_date":           event_date,
            "feature_used":         random.choice(modules),
            "session_duration_min": session_duration_min,
            "login_count":          random.randint(1,5),
            "actions_count":        actions_count,
            "device_type":          random.choice(["desktop","mobile","tablet"])
        })

    return logs

In [16]:
logs = [generar_logs_cliente(customer) for customer in df_customers.to_dict('records')]

In [17]:
logs_flatten = [log for cliente_logs in logs for log in cliente_logs]

In [18]:
df_logs = pd.DataFrame(logs_flatten)
df_logs

,log_id,customer_id,user_id,event_date,feature_used,session_duration_min,login_count,actions_count,device_type
0,9373c9ca-ddd5-45c4-ab65-0355efa4102c,9dc8e7a4-7c12-42c8-9397-ebdb9348630b,0270fa7f-0104-4c41-ac38-42769e7afde7,2025-10-02,billing,15,2,3,mobile
1,f8eff719-4a1c-458b-9028-fd3ec663bbd9,9dc8e7a4-7c12-42c8-9397-ebdb9348630b,09c50d68-02f2-4f32-abb4-93b82ca25025,2025-09-18,dashboard,26,4,5,tablet
2,53e0be08-8068-4cb2-b80a-3d0a5101b6f6,9dc8e7a4-7c12-42c8-9397-ebdb9348630b,816afed0-93fc-4c7e-952e-d1f62b1b4d80,2025-10-04,dashboard,30,2,21,desktop
3,68d2b75c-696c-4697-9ac9-05fc2b5149a3,9dc8e7a4-7c12-42c8-9397-ebdb9348630b,816afed0-93fc-4c7e-952e-d1f62b1b4d80,2025-09-08,billing,15,1,18,tablet
4,73e4869f-fdc1-4b86-885b-0a3227516680,9dc8e7a4-7c12-42c8-9397-ebdb9348630b,816afed0-93fc-4c7e-952e-d1f62b1b4d80,2025-10-02,integrations,29,5,27,desktop
...,...,...,...,...,...,...,...,...,...
179414,ae60cbf4-ec2b-4fb0-ad73-e0f7f86d310f,f6bb20a7-30cf-4ec7-9b89-19d1deb1e032,faa1f9a2-cfac-4653-b01d-1828f4bbb632,2024-02-24,billing,10,4,49,tablet
179415,fec6d4e1-46df-48c6-a4e6-ed703530a5e3,f6bb20a7-30cf-4ec7-9b89-19d1deb1e032,05d5cdbd-e39c-4293-b157-9ca590d0c4e3,2023-12-31,notifications,43,3,16,mobile
179416,81d1d6f7-2e12-4dd3-b90c-cdccb6e8615e,f6bb20a7-30cf-4ec7-9b89-19d1deb1e032,f9763e75-c0b8-4b65-94f5-6f8204776913,2023-10-22,calendar,115,2,10,mobile
179417,42924bbc-df0d-40ec-8355-3e1de3bb1130,f6bb20a7-30cf-4ec7-9b89-19d1deb1e032,78dd6749-2d52-494b-bcb7-0c2e5518de7e,2023-08-27,calendar,60,5,35,mobile


In [19]:
l_logs = int(len(df_logs))
indices = np.random.choice(l_logs, size=int(l_logs*0.03), replace=False)
df_logs.loc[indices, "session_duration_min"] = np.nan

In [20]:
df_logs.isna().sum()

log_id                     0
customer_id                0
user_id                    0
event_date                 0
feature_used               0
session_duration_min    5382
login_count                0
actions_count              0
device_type                0
dtype: int64

In [21]:
df_logs.to_csv('../data/raw/usage_logs.csv', index=False)

## Support Tickets
- ticket_id
- customer_id
- created_at
- resolved_at
- category
- priority
- sentiment_score
- csat_score
- escalated

In [22]:
def generar_tickets(cliente: dict):
    if cliente["churned"] == True:
        tickets_count = random.randint(5,21)
    else:
        tickets_count = random.randint(1,9)

    tickets = []
    for _ in range(tickets_count):
        create_at = fake.date_between(start_date=cliente["contract_start"], end_date=cliente["contract_end"])
        if cliente["churned"] == True:
            r = random.randint(0,100)
            if r > 70:
                resolved_at = fake.date_between(start_date=create_at, end_date=cliente["contract_end"])
            elif r <= 70:
                resolved_at = None
            
            sentiment_score = random.uniform(-1,0.2)

            csat_score = random.randint(1,3)

            r = random.randint(0,100)

            if r > 60:
                escalated  = True
            elif r <= 60:
                escalated  = False

        else:
            r = random.randint(0,100)
            if r > 85:
                resolved_at = None
            elif r <= 85:
                resolved_at = fake.date_between(start_date=create_at, end_date=cliente["contract_end"])
            
            sentiment_score = random.uniform(-0.2, 1)

            csat_score = random.randint(2,5)

            r = random.randint(0,100)

            if r > 90:
                escalated  = True
            elif r <= 90:
                escalated  = False

        tickets.append(
        {
            "ticket_id":        fake.uuid4(),
            "customer_id":      cliente["customer_id"],
            "create_at":        create_at,
            "resolved_at":      resolved_at,
            "category":         random.choice(["billing", "technical", "onboarding", "feature_request", "bug"]),
            "priority":         random.choice(["low", "medium", "high", "critical"]),
            "sentiment_score":  sentiment_score,
            "csat_score":       csat_score,
            "escalated":        escalated
        }
        )
    
    return tickets


In [23]:
tickets = [generar_tickets(customer) for customer in df_customers.to_dict('records')]

In [24]:
tickets_flatten = [ticket for tickets_cust in tickets for ticket in tickets_cust]

In [25]:
df_tickets = pd.DataFrame(tickets_flatten)
df_tickets

,ticket_id,customer_id,create_at,resolved_at,category,priority,sentiment_score,csat_score,escalated
0,33e69ec5-220c-451b-bb44-2d8d8ad9478e,9dc8e7a4-7c12-42c8-9397-ebdb9348630b,2025-10-25,None,onboarding,critical,-0.223063,1,False
1,a11b7966-3b79-49c0-a04b-9dc3a4517e94,9dc8e7a4-7c12-42c8-9397-ebdb9348630b,2025-10-29,2025-11-01,technical,critical,-0.572368,3,False
2,5da20066-c8e8-423e-96b0-6d8022e7820e,9dc8e7a4-7c12-42c8-9397-ebdb9348630b,2025-08-23,None,billing,high,-0.911469,3,True
3,c15f8c19-18de-4150-8add-229a32ea407e,9dc8e7a4-7c12-42c8-9397-ebdb9348630b,2025-10-26,None,feature_request,critical,0.063601,1,False
4,690999db-230d-4b17-ab2e-721d433e7b39,9dc8e7a4-7c12-42c8-9397-ebdb9348630b,2025-10-15,2025-10-22,technical,critical,-0.577904,2,False
...,...,...,...,...,...,...,...,...,...
27354,b73c4b0a-1ebb-4eee-9297-25ece99a20d6,f6bb20a7-30cf-4ec7-9b89-19d1deb1e032,2024-03-06,2024-03-09,feature_request,high,0.451960,3,False
27355,fe8b083f-fc74-4c56-ac2d-aa63ef4eaded,f6bb20a7-30cf-4ec7-9b89-19d1deb1e032,2024-01-26,2024-04-06,onboarding,critical,0.258837,4,False
27356,33d47228-0b09-4681-b1be-af40420f31d0,f6bb20a7-30cf-4ec7-9b89-19d1deb1e032,2024-05-30,None,bug,high,0.727776,3,False
27357,92f1d9f0-72a3-4ce7-bab0-f01f177504f9,f6bb20a7-30cf-4ec7-9b89-19d1deb1e032,2024-01-18,2024-03-07,feature_request,critical,0.270478,5,True


In [26]:
l_tickets = int(len(df_tickets))
indices = np.random.choice(l_tickets, size=int(l_tickets*0.15), replace=False)
df_tickets.loc[indices, "csat_score"] = np.nan

In [27]:
indices = np.random.choice(l_tickets, size=int(l_tickets*0.10), replace=False)
df_tickets.loc[indices, "sentiment_score"] = np.nan

In [28]:
df_tickets.isna().sum()

ticket_id              0
customer_id            0
create_at              0
resolved_at        15111
category               0
priority               0
sentiment_score     2735
csat_score          4103
escalated              0
dtype: int64

In [29]:
df_tickets.to_csv('../data/raw/support_tickets.csv', index = False)